# **# API Enrichment: Weather + Currency**

### **Weather Enrichment**

In [1]:
# HappyBooking - Step 2: API Enrichment (Weather + Currency)
#
# We pull a sample of distinct cities from the Bronze batch table
# to use as input for the weather API lookup. We filter out obviously
# dirty values (placeholders, empty strings) using a simple pattern,
# since we already know this dataset contains such noise.

cities_df = spark.sql("""
    SELECT DISTINCT city
    FROM bronze_hotel_booking_batch
    WHERE city IS NOT NULL
      AND TRIM(city) != ''
      AND city NOT RLIKE '[?_!]'
    LIMIT 20
""")

cities = [row["city"].strip() for row in cities_df.collect()]
print(f"Selected {len(cities)} cities:")
print(cities)

StatementMeta(, 89f684ca-711f-4cf3-a533-a36d42664573, 3, Finished, Available, Finished, False)

Selected 20 cities:
['Da Nang', 'Bangalore', 'Antwerp', 'Thessaloniki', 'Cape town', 'Viña del mar', 'plovdiv', 'Belgrade', 'Linz', 'Budapest', 'Ostrava', 'PATRAS', 'Incheon', 'Rotterdam...', 'Arequipa', 'Casablanca', 'Cairo', 'Brasov', 'Abuja', 'Wellington']


In [3]:
import requests
import time

def geocode_city(city_name, retries=2):
    """
    Resolve a city name to latitude/longitude using Open-Meteo's
    free geocoding API. Returns None if the city can't be resolved
    or the request fails after retries - this can happen with noisy
    city names in our dataset, or transient network issues.
    """
    url = "https://geocoding-api.open-meteo.com/v1/search"
    params = {"name": city_name, "count": 1}

    for attempt in range(retries + 1):
        try:
            response = requests.get(url, params=params, timeout=20)
            data = response.json()
            if "results" not in data or len(data["results"]) == 0:
                return None
            result = data["results"][0]
            return {
                "resolved_name": result["name"],
                "country": result.get("country"),
                "latitude": result["latitude"],
                "longitude": result["longitude"],
            }
        except requests.exceptions.RequestException as e:
            if attempt == retries:
                print(f"  Failed to geocode '{city_name}' after {retries + 1} attempts: {e}")
                return None
            time.sleep(1)  # brief pause before retry


def get_weather(latitude, longitude, retries=2):
    """
    Fetch current weather for a given coordinate using Open-Meteo's
    free weather API. No API key required.
    """
    url = "https://api.open-meteo.com/v1/forecast"
    params = {
        "latitude": latitude,
        "longitude": longitude,
        "current": "temperature_2m,relative_humidity_2m,precipitation,weather_code",
    }

    for attempt in range(retries + 1):
        try:
            response = requests.get(url, params=params, timeout=20)
            return response.json().get("current", {})
        except requests.exceptions.RequestException as e:
            if attempt == retries:
                print(f"  Failed to get weather for ({latitude}, {longitude}): {e}")
                return {}
            time.sleep(1)


weather_records = []

for city in cities:
    geo = geocode_city(city)
    if geo is None:
        print(f"Could not resolve city: {city}")
        continue

    weather = get_weather(geo["latitude"], geo["longitude"])

    weather_records.append({
        "queried_city": city,
        "resolved_city": geo["resolved_name"],
        "country": geo["country"],
        "latitude": geo["latitude"],
        "longitude": geo["longitude"],
        "temperature_c": weather.get("temperature_2m"),
        "humidity_pct": weather.get("relative_humidity_2m"),
        "precipitation_mm": weather.get("precipitation"),
        "weather_code": weather.get("weather_code"),
    })

    time.sleep(0.3)

print(f"\nSuccessfully enriched {len(weather_records)} out of {len(cities)} cities.")

StatementMeta(, 89f684ca-711f-4cf3-a533-a36d42664573, 5, Finished, Available, Finished, False)

  Failed to get weather for (37.80228, -6.70303): HTTPSConnectionPool(host='api.open-meteo.com', port=443): Read timed out. (read timeout=20)
Could not resolve city: Rotterdam...

Successfully enriched 19 out of 20 cities.


In [4]:
# Convert the enriched weather records into a Spark DataFrame and
# write them as a Bronze table. Same pattern as our other Bronze
# tables: raw, minimally processed, ready for Silver-layer cleaning.

df_weather = spark.createDataFrame(weather_records)
df_weather.show(truncate=40)

df_weather.write.format("delta").mode("overwrite").saveAsTable("bronze_weather_enrichment")
print("Bronze table 'bronze_weather_enrichment' created successfully.")

StatementMeta(, 89f684ca-711f-4cf3-a533-a36d42664573, 6, Finished, Available, Finished, False)

+------------+------------+---------+---------+----------------+------------+--------------+-------------+------------+
|     country|humidity_pct| latitude|longitude|precipitation_mm|queried_city| resolved_city|temperature_c|weather_code|
+------------+------------+---------+---------+----------------+------------+--------------+-------------+------------+
|     Vietnam|          63| 16.06778|108.22083|             0.0|     Da Nang|       Da Nang|         30.7|           2|
|    Pakistan|          70|  24.8717|  67.0839|             0.0|   Bangalore|Bangalore Town|         29.3|           3|
|     Belgium|          62| 51.22047|  4.40026|             0.0|     Antwerp|       Antwerp|         20.5|           2|
|      Greece|          44| 40.64072| 22.93493|             0.0|Thessaloniki|  Thessaloniki|         33.0|           3|
|South Africa|          59|-33.92584| 18.42322|             0.0|   Cape town|     Cape Town|         20.1|           3|
|       Chile|          93|-33.02457|-71

### **Currency Enrichment**

In [5]:
import requests

def get_exchange_rates(base_currency="EUR", retries=2):
    """
    Fetch current exchange rates for a set of major currencies
    relative to a base currency, using the free Frankfurter API
    (backed by European Central Bank reference rates, no API key
    required).
    """
    url = "https://api.frankfurter.dev/v1/latest"
    params = {"base": base_currency}

    for attempt in range(retries + 1):
        try:
            response = requests.get(url, params=params, timeout=20)
            data = response.json()
            return data
        except requests.exceptions.RequestException as e:
            if attempt == retries:
                print(f"Failed to fetch exchange rates: {e}")
                return None
            time.sleep(1)


rate_data = get_exchange_rates(base_currency="EUR")

if rate_data:
    currency_records = [
        {
            "base_currency": rate_data["base"],
            "quote_currency": currency,
            "rate": rate,
            "rate_date": rate_data["date"],
        }
        for currency, rate in rate_data["rates"].items()
    ]

    print(f"Fetched {len(currency_records)} exchange rates as of {rate_data['date']}")
    for record in currency_records[:5]:
        print(record)
else:
    currency_records = []
    print("No exchange rate data retrieved.")

StatementMeta(, 89f684ca-711f-4cf3-a533-a36d42664573, 7, Finished, Available, Finished, False)

Fetched 29 exchange rates as of 2026-09-01
{'base_currency': 'EUR', 'quote_currency': 'AUD', 'rate': 1.623, 'rate_date': '2026-09-01'}
{'base_currency': 'EUR', 'quote_currency': 'BRL', 'rate': 6.0255, 'rate_date': '2026-09-01'}
{'base_currency': 'EUR', 'quote_currency': 'CAD', 'rate': 1.6096, 'rate_date': '2026-09-01'}
{'base_currency': 'EUR', 'quote_currency': 'CHF', 'rate': 0.9394, 'rate_date': '2026-09-01'}
{'base_currency': 'EUR', 'quote_currency': 'CNY', 'rate': 7.7911, 'rate_date': '2026-09-01'}


In [6]:
# Convert the currency records into a Spark DataFrame and write
# them as a Bronze table, following the same raw-landing pattern
# as our other Bronze sources.

df_currency = spark.createDataFrame(currency_records)
df_currency.show(30, truncate=False)

df_currency.write.format("delta").mode("overwrite").saveAsTable("bronze_currency_enrichment")
print("Bronze table 'bronze_currency_enrichment' created successfully.")

StatementMeta(, 89f684ca-711f-4cf3-a533-a36d42664573, 8, Finished, Available, Finished, False)

+-------------+--------------+--------+----------+
|base_currency|quote_currency|rate    |rate_date |
+-------------+--------------+--------+----------+
|EUR          |AUD           |1.623   |2026-09-01|
|EUR          |BRL           |6.0255  |2026-09-01|
|EUR          |CAD           |1.6096  |2026-09-01|
|EUR          |CHF           |0.9394  |2026-09-01|
|EUR          |CNY           |7.7911  |2026-09-01|
|EUR          |CZK           |24.159  |2026-09-01|
|EUR          |DKK           |7.4748  |2026-09-01|
|EUR          |GBP           |0.85655 |2026-09-01|
|EUR          |HKD           |9.0877  |2026-09-01|
|EUR          |HUF           |366.71  |2026-09-01|
|EUR          |IDR           |20566.11|2026-09-01|
|EUR          |ILS           |3.4938  |2026-09-01|
|EUR          |INR           |110.0485|2026-09-01|
|EUR          |ISK           |140.8   |2026-09-01|
|EUR          |JPY           |185.63  |2026-09-01|
|EUR          |KRW           |1593.17 |2026-09-01|
|EUR          |MXN           |1